# GPT-4o Few-Shot Prediction of Treatment Response

Few-shot classification of responder vs. non-responder to a smartphone-based
mindfulness intervention in autistic adults, using a compact 7-feature
tokenized representation of baseline measures.

**Outcome definition:** Responder (label 1) = STAI-State decrease of >= 7 points
from pre- to post-intervention. Non-responder (label 0) = decrease < 7 points.

**Procedure:** For each shot count N in {20, 30, 40, 50, 60, 70}, we draw N
labeled examples at random as the few-shot prompt and classify all remaining
participants. This is repeated over 5 random draws (seeds 42-46) and metrics
are averaged across repeats.

**Input:** `X_token_tokenized.csv` with columns `text` (tokenized baseline
features) and `label` (0/1). The dataset contains individual-level clinical
trial data and is **not** included in this repository. Access is governed by
the original trial's data sharing agreement.

**Requirements:** Set the `OPENAI_API_KEY` environment variable before running.
Do not hard-code API keys in this notebook.

## 1. Setup

In [ ]:
# pip install --upgrade openai pandas numpy scikit-learn

import os
import re
import json
import time

import numpy as np
import pandas as pd
from openai import OpenAI
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

# ---- Configuration ----
# Model reported in the manuscript. Override with OPENAI_MODEL if needed.
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")
TEMPERATURE = 0.0

SHOT_COUNTS = [20, 30, 40, 50, 60, 70]
N_REPEATS = 5
BASE_SEED = 42

MAX_RETRIES = 3
RETRY_BACKOFF = 2.0

CSV_PATH = os.getenv("DATA_CSV", "X_token_tokenized.csv")
OUT_DIR = "results"

client = OpenAI()
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Load data

In [ ]:
df = pd.read_csv(CSV_PATH)

assert {"text", "label"}.issubset(df.columns), \
    "CSV must contain 'text' and 'label' columns"


def to_binary(x) -> int:
    #Coerce a raw label value to 0/1.
    try:
        return 1 if int(str(x).strip()) == 1 else 0
    except (TypeError, ValueError):
        return 0


df["_label"] = df["label"].map(to_binary)

print(f"n = {len(df)}  |  responders = {df['_label'].sum()}  "
      f"|  non-responders = {(1 - df['_label']).sum()}")

## 3. Prompt construction and inference

In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical evaluator determining whether an autistic adult showed a "
    "meaningful treatment response after a mindfulness-based intervention.\n"
    "\n"
    "Classification rule:\n"
    "- Responder (label 1): STAI state anxiety decreased by >= 7 points.\n"
    "- Non-responder (label 0): decrease < 7.\n"
    "\n"
    "You will receive the following baseline variables "
    "(higher levels indicate higher anxiety unless noted):\n"
    "- STAI_17: 'I feel calm.'\n"
    "- STAI_1:  'I feel strained.'\n"
    "- STAI_4:  'I am worrying over possible misfortunes.'\n"
    "- STAI_13: 'I am jittery.'\n"
    "- STAI_7:  'I am worried.'\n"
    "- AQ40:    'When I was young, I enjoyed pretend-play with other children.' "
    "(Higher levels = stronger endorsement.)\n"
    "- Age: numeric (years).\n"
    "\n"
    "The input will list these variables in text form (e.g., 'STAI_17: moderately so'). "
    "Use only the provided variables and the few-shot examples given by the user to infer "
    "whether the pattern indicates a responder (1) or a non-responder (0) under the "
    ">= 7-point rule.\n"
    "\n"
    "Think internally but do NOT reveal any reasoning.\n"
    "Return ONLY valid JSON: {\"label\": 0 or 1}."
)


def build_few_shot_block(fewshot_df: pd.DataFrame) -> str:
    # Render the sampled examples as a single prompt block
    return "\n\n".join(
        f"INPUT:\n{row['text']}\nLABEL: {int(row['_label'])}"
        for _, row in fewshot_df.iterrows()
    )


def parse_label(text: str):
    # Extract a 0/1 label from the model response. Returns None if unparseable.
    text = (text or "").strip()

    # 1) Direct JSON parse
    try:
        value = int(json.loads(text).get("label"))
        return 1 if value == 1 else 0
    except (ValueError, TypeError, AttributeError, json.JSONDecodeError):
        pass

    # 2) Regex fallback for JSON-like output
    match = re.search(r'"label"\s*:\s*"?([01])"?', text)
    if match:
        return int(match.group(1))

    # 3) Bare 0/1 response
    if text in ("0", "1"):
        return int(text)

    return None

In [ ]:
def classify_one(text: str, few_shot_block: str, n_shots: int):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Here are {n_shots} examples:\n\n{few_shot_block}"
                f"\n\nNow classify this.\nINPUT:\n{text}"
            ),
        },
    ]

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                messages=messages,
                response_format={"type": "json_object"},
            )
            return parse_label(response.choices[0].message.content)
        except Exception as exc:  # noqa: BLE001 - log and retry
            if attempt == MAX_RETRIES - 1:
                print(f"    API call failed after {MAX_RETRIES} attempts: {exc}")
                return None
            time.sleep(RETRY_BACKOFF * (attempt + 1))

    return None

## 4. Run the few-shot sweep

For each shot count, `N_REPEATS` random draws are evaluated. Samples whose API
call failed or whose response could not be parsed are excluded from the metrics
and reported separately in `n_failed`.

In [ ]:
def run_one_repeat(df: pd.DataFrame, n_shots: int, seed: int) -> tuple:

    fewshot = df.sample(n_shots, random_state=seed)
    test = df.drop(fewshot.index).copy()
    few_shot_block = build_few_shot_block(fewshot)

    predictions = []
    for i, text in enumerate(test["text"], start=1):
        if i % 10 == 0:
            print(f"    inference {i}/{len(test)}")
        predictions.append(classify_one(text, few_shot_block, n_shots))

    test["pred"] = predictions
    test["true"] = test["_label"]

    n_failed = int(test["pred"].isna().sum())
    scored = test.dropna(subset=["pred"])

    y_true = scored["true"].to_numpy()
    y_pred = scored["pred"].astype(int).to_numpy()

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    summary = {
        "n_shots": n_shots,
        "seed": seed,
        "n_test": len(test),
        "n_scored": len(scored),
        "n_failed": n_failed,
        "acc": accuracy_score(y_true, y_pred),
        "prec_0": precision[0], "rec_0": recall[0], "f1_0": f1[0],
        "prec_1": precision[1], "rec_1": recall[1], "f1_1": f1[1],
        "cm_00": cm[0, 0], "cm_01": cm[0, 1],
        "cm_10": cm[1, 0], "cm_11": cm[1, 1],
    }

    preds_out = test[["true", "pred"]].copy()
    preds_out["n_shots"] = n_shots
    preds_out["seed"] = seed
    preds_out["row_index"] = test.index

    return summary, preds_out

In [ ]:
all_summaries = []
all_predictions = []

overall_start = time.time()

for n_shots in SHOT_COUNTS:
    assert len(df) > n_shots, f"Not enough rows for {n_shots} shots"
    print(f"=== {n_shots}-shot ===")

    for repeat in range(N_REPEATS):
        seed = BASE_SEED + repeat
        print(f"  repeat {repeat + 1}/{N_REPEATS} (seed={seed})")
        start = time.time()

        summary, preds = run_one_repeat(df, n_shots, seed)

        all_summaries.append(summary)
        all_predictions.append(preds)

        print(f"  done in {time.time() - start:.1f}s  |  "
              f"acc={summary['acc']:.3f}  failed={summary['n_failed']}")

    print()

summary_df = pd.DataFrame(all_summaries)
predictions_df = pd.concat(all_predictions, ignore_index=True)

print(f"Total runtime: {(time.time() - overall_start) / 60:.1f} min")

## 5. Results

Note: predictions are saved without the raw `text` field so that no
participant-level feature values are written to disk alongside the outputs.

In [ ]:
metric_cols = ["acc", "prec_0", "rec_0", "f1_0", "prec_1", "rec_1", "f1_1"]

by_shots = (
    summary_df
    .groupby("n_shots")[metric_cols + ["n_failed"]]
    .agg(["mean", "std"])
    .round(4)
)

print(by_shots["acc"])

summary_df.to_csv(f"{OUT_DIR}/fewshot_summary.csv", index=False)
predictions_df.to_csv(f"{OUT_DIR}/fewshot_predictions.csv", index=False)
by_shots.to_csv(f"{OUT_DIR}/fewshot_by_shots.csv")

print(f"\nSaved to {OUT_DIR}/")